# Model author: declare and inspect one simple resonator

This Notebook is for the person who owns the reusable circuit model. It starts at Library components, composes their pins into Public or anonymous Internal electric nodes, attaches one logical Port, and then derives response and quantity views from the sealed `CircuitPlan`. It teaches response solving and physical-quantity evaluation separately, then uses both Results together. Open this same `.ipynb` in VS Code to execute it or on GitHub to read its static rendered form. At the current `CONVERGING` checkpoint the package is an API-only scaffold, so executing a construction cell intentionally raises `ScaffoldUnavailableError`.

## Choose the operation before the Spec

| Goal | Call | Spec |
|---|---|---|
| Inspect S/Y/Z over a grid | `run.solve(view, spec)` | `DirectSolveSpec` |
| Evaluate one Direct physical quantity | `run.evaluate(view, spec)` | `DiagonalRootSpec` or another typed quantity Spec |
| Materialize the complete Direct operator | `run.evaluate(view, spec)` | `OperatorSpec` |

A solve-spec class requests a response surface. V1 has no separate `SolveSpec` class: construct `DirectSolveSpec` or `HBSolveSpec` directly. A quantity Spec requests its named physical quantity without an unrelated S/Y/Z sweep. `show()` only presents an existing Result and never executes again.

In [ ]:
from scnsim import (
    CircuitPlan, CircuitRun, DiagonalRootSpec, DirectSolveSpec,
    ReductionPipeline, ReportSpec, SParameterTrace,
    library as sc, units as u,
)

## 1. The Plan is the physical authority

Components come from an exact Library. One `plan.net(...)` call assigns all pins of an equipotential electric node. Supplying `id=` makes a Public coordinate; omitting it makes an anonymous Internal connection that still participates in compilation. A logical Port is a special Plan component attached through the returned `ElectricNodeRef`, never directly to a pin. The two serial 12 fF capacitors preserve the former 6 fF series equivalent while making one genuine Internal node visible in the authoring flow.

In [ ]:
plan = CircuitPlan(id="simple_readout")

input_cap = plan.add(
    sc.capacitor(id="input_cap", capacitance=12.0 * u.fF)
)
coupling_cap = plan.add(
    sc.capacitor(id="coupling_cap", capacitance=12.0 * u.fF)
)
readout = plan.add(
    sc.grounded_parallel_linear_lc_resonator(
        id="readout",
        subsystem_capacitance=110.0 * u.fF,
        inductance=5.8 * u.nH,
    )
)

plan.reference("ground")
signal_boundary = plan.net(input_cap.pin("a"))
plan.net(input_cap.pin("b"), coupling_cap.pin("a"))
plan.net(
    coupling_cap.pin("b"),
    readout.pin("signal"),
    id="readout_node",
)
signal_port = plan.add_port(
    id="signal_in",
    at=signal_boundary,
    role="terminated",
    reference_impedance=50.0 * u.ohm,
)

## 2. Solve a response surface with `DirectSolveSpec`

`run.original` is the full Plan reference. `retain("signal_in")` selects the Public node promoted by the logical Port, so `run.solve(response_view, direct_spec)` can publish the complete selected-view S/Y/Z response. The node ID, Port ID, and trace channel happen to share `"signal_in"` only because anonymous-node promotion deliberately adopts the Port ID. If the node had been explicitly named `"signal_node"` while its Port remained `"signal_in"`, both `retain(...)` and `SParameterTrace.input_port/output_port` would use `"signal_node"`, while `CurrentDrive.at` would use the returned PortRef. This Plan has one external Port, so its S family is the 1-by-1 S11 response; SCNSim does not guess S21. A trace is a projection of that complete matrix and does not launch another solve. `run.explain(...)` is the preflight inspection point.

In [ ]:
run = CircuitRun(plan=plan, workspace="results/simple_readout")
response_view = run.original.reduce(
    ReductionPipeline().retain("signal_in")
)
quantity_view = run.original.reduce(
    ReductionPipeline().retain("readout_node")
)
frequency_grid = tuple(
    value * u.GHz
    for value in (5.5, 5.6, 5.7, 5.8, 5.9, 6.0, 6.1, 6.2, 6.3)
)

direct_spec = DirectSolveSpec(
    frequencies=frequency_grid,
    traces=(
        SParameterTrace(
            id="reflection",
            input_port="signal_in", input_mode=(),
            output_port="signal_in", output_mode=(),
        ),
    ),
)
run.explain(response_view, direct_spec).show()
direct = run.solve(response_view, direct_spec)
direct.s.show(magnitude="db")
direct.traces["reflection"].show(magnitude="db")

## 3. Evaluate one physical quantity with `DiagonalRootSpec`

`run.evaluate(quantity_view, readout_root)` may retain the named unported Public `readout_node` because evaluation does not promise a wave-port response. It solves for one simple complex root of that dynamic-operator diagonal. Required `root_hint` only locates the branch at the sealed baseline; it is not the returned root, a target, a constraint, or a search window. Later ParameterSets continue the baseline branch instead of choosing the root nearest the hint again. This evaluation does not calculate the preceding frequency-grid S/Y/Z response. A coupled retained block would use `HybridizedPoleSpec`, while the full labeled matrix would use `OperatorSpec`.

In [ ]:
readout_root = DiagonalRootSpec(
    coordinate="readout_node",
    root_hint=6.0 * u.GHz,
)
evaluated = run.evaluate(quantity_view, readout_root)
evaluated.show()
evaluated.frequency.to(u.GHz)

## 4. Use the response and physical quantity together

The response Result and root Result remain separate exact requests, but downstream Python and reporting can use both. `ReportSpec` neither solves nor searches for a latest result; it records exactly which Results it presents.

In [ ]:
response_grid = direct.frequencies.to(u.GHz)
root_frequency = evaluated.frequency.to(u.GHz)
response_grid, root_frequency

report = run.build_report(
    ReportSpec(inputs=(direct, evaluated))
)
report.show()